In [ ]:
!apt-get update -qq
!apt-get install -y default-jre-headless -qq

!pip install -q datasets transformers arabert farasapy scikit-learn

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
import pandas as pd
import numpy as np

from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from arabert.preprocess import ArabertPreprocessor

In [ ]:
dataset = load_dataset(
    "Qanadil/ArSAS_An_Arabic_Speech-Act_and_Sentiment_Corpus_of_Tweets"
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['#Tweet_ID', 'Tweet_text', 'Topic', 'Sentiment_label', 'Sentiment_label_confidence', 'Speech_act_label', 'Speech_act_label_confidence', '__index_level_0__'],
        num_rows: 18678
    })
})


In [ ]:
print("Dataset columns:")
print(dataset["train"].column_names)

print("\nFirst example:")
print(dataset["train"][0])

print("\nNumber of examples:")
print(len(dataset["train"]))

Dataset columns:
['#Tweet_ID', 'Tweet_text', 'Topic', 'Sentiment_label', 'Sentiment_label_confidence', 'Speech_act_label', 'Speech_act_label_confidence', '__index_level_0__']

First example:
{'#Tweet_ID': 929241870508724224, 'Tweet_text': 'المباراة القـادمة #غانا x #مصر الجولة الأخيرة من المجموعة الـ 5 تصفيات كاس العالم 2018 روسـيا ترتيب مصر : المركز الاول 12 نقطة ( تم حسم التأهل للمونديال ) غــدا الساعة 5:30 ع قناة : بين ســبورت 1 تـــوقعاتكم لـ نتيجة الماتش .؟ 😀😁 https://t.co/RTQBNZXDqM', 'Topic': 'Event', 'Sentiment_label': 'Positive', 'Sentiment_label_confidence': 0.38, 'Speech_act_label': 'Assertion', 'Speech_act_label_confidence': 0.62, '__index_level_0__': 0}

Number of examples:
18678


In [ ]:
df = dataset["train"].to_pandas()

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (18678, 8)


,#Tweet_ID,Tweet_text,Topic,Sentiment_label,Sentiment_label_confidence,Speech_act_label,Speech_act_label_confidence,__index_level_0__
0,929241870508724224,المباراة القـادمة #غانا x #مصر الجولة الأخيرة ...,Event,Positive,0.38,Assertion,0.62,0
1,928942264583376897,هل هذه هي سياسة خارجيه لدوله تحترم نفسها والآخ...,Entity,Negative,1.00,Expression,0.68,1
2,928615163250520065,وزير خارجية فرنسا عن منتدى شباب العالم: شعرت ب...,Event,Positive,0.69,Assertion,1.00,2
3,931614713368186880,ومع السيسي و بشار و ايران و بن زايد و والا خلي...,Event,Negative,1.00,Expression,1.00,3
4,929755693011427331,أهداف مباراة غانا 0 مصر 1 تصفيات كأس العالم 20...,Event,Neutral,1.00,Assertion,1.00,4


In [ ]:
print("Columns in the dataset:")
print(df.columns.tolist())

Columns in the dataset:
['#Tweet_ID', 'Tweet_text', 'Topic', 'Sentiment_label', 'Sentiment_label_confidence', 'Speech_act_label', 'Speech_act_label_confidence', '__index_level_0__']


In [ ]:
text_candidates = [
    "tweet_text",
    "Tweet_text",
    "Tweet_Text",
    "tweet",
    "Tweet",
    "text",
    "Text"
]

text_column = None

for col in text_candidates:
    if col in df.columns:
        text_column = col
        break

if text_column is None:
    raise ValueError(
        "Tweet text column was not found. Available columns are: "
        + str(df.columns.tolist())
    )

print("Tweet text column:", text_column)
print("Sentiment column: Sentiment_label")

Tweet text column: Tweet_text
Sentiment column: Sentiment_label


In [ ]:
df = df[[text_column, "Sentiment_label"]].copy()

df = df.rename(
    columns={
        text_column: "tweet_text"
    }
)

df.head()

,tweet_text,Sentiment_label
0,المباراة القـادمة #غانا x #مصر الجولة الأخيرة ...,Positive
1,هل هذه هي سياسة خارجيه لدوله تحترم نفسها والآخ...,Negative
2,وزير خارجية فرنسا عن منتدى شباب العالم: شعرت ب...,Positive
3,ومع السيسي و بشار و ايران و بن زايد و والا خلي...,Negative
4,أهداف مباراة غانا 0 مصر 1 تصفيات كأس العالم 20...,Neutral


In [ ]:
print("Sentiment classes:")
print(df["Sentiment_label"].value_counts())

print("\nUnique labels:")
print(df["Sentiment_label"].unique())

print("\nNumber of sentiment classes:")
print(df["Sentiment_label"].nunique())

Sentiment classes:
Sentiment_label
Negative    7384
Neutral     6894
Positive    4400
Name: count, dtype: int64

Unique labels:
['Positive' 'Negative' 'Neutral']

Number of sentiment classes:
3


In [ ]:
print("Missing values before cleaning:")
print(df.isnull().sum())

Missing values before cleaning:
tweet_text         0
Sentiment_label    0
dtype: int64


In [ ]:
print("Number of duplicate tweets before cleaning:")
print(df["tweet_text"].duplicated().sum())

Number of duplicate tweets before cleaning:
88


In [ ]:
# Remove rows with missing tweet text or sentiment labels
df = df.dropna(
    subset=["tweet_text", "Sentiment_label"]
).copy()

# Convert tweet text to string
df["tweet_text"] = df["tweet_text"].astype(str)

# Remove extra spaces
df["tweet_text"] = df["tweet_text"].str.strip()

# Remove empty tweets
df = df[df["tweet_text"] != ""].copy()

# Remove duplicate tweets
df = df.drop_duplicates(
    subset=["tweet_text"]
).copy()

# Reset index
df = df.reset_index(drop=True)

print("Dataset size after cleaning:", len(df))

Dataset size after cleaning: 18590


In [ ]:
print("Missing values after cleaning:")
print(df.isnull().sum())

print("\nRemaining duplicate tweets:")
print(df["tweet_text"].duplicated().sum())

print("\nEmpty tweets:")
print((df["tweet_text"] == "").sum())

Missing values after cleaning:
tweet_text         0
Sentiment_label    0
dtype: int64

Remaining duplicate tweets:
0

Empty tweets:
0


In [ ]:
print("Sentiment distribution after cleaning:")
print(df["Sentiment_label"].value_counts())

print("\nPercentage distribution:")
print(
    df["Sentiment_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Sentiment distribution after cleaning:
Sentiment_label
Negative    7340
Neutral     6870
Positive    4380
Name: count, dtype: int64

Percentage distribution:
Sentiment_label
Negative    39.48
Neutral     36.96
Positive    23.56
Name: proportion, dtype: float64


In [ ]:
# First split:
# 80% training and 20% temporary data

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["Sentiment_label"]
)

# Second split:
# Split the remaining 20% equally:
# 10% validation and 10% test

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["Sentiment_label"]
)

# Reset indices

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [ ]:
print("Training examples:", len(train_df))
print("Validation examples:", len(validation_df))
print("Test examples:", len(test_df))

print(
    "\nTotal examples:",
    len(train_df)
    + len(validation_df)
    + len(test_df)
)

Training examples: 14872
Validation examples: 1859
Test examples: 1859

Total examples: 18590


In [ ]:
print("TRAIN DISTRIBUTION")
print(train_df["Sentiment_label"].value_counts())

print("\nVALIDATION DISTRIBUTION")
print(validation_df["Sentiment_label"].value_counts())

print("\nTEST DISTRIBUTION")
print(test_df["Sentiment_label"].value_counts())

TRAIN DISTRIBUTION
Sentiment_label
Negative    5872
Neutral     5496
Positive    3504
Name: count, dtype: int64

VALIDATION DISTRIBUTION
Sentiment_label
Negative    734
Neutral     687
Positive    438
Name: count, dtype: int64

TEST DISTRIBUTION
Sentiment_label
Negative    734
Neutral     687
Positive    438
Name: count, dtype: int64


In [ ]:
model_name = "aubmindlab/bert-base-arabertv02"

arabert_prep = ArabertPreprocessor(
    model_name=model_name
)

print("AraBERT preprocessor initialized successfully.")
print("Model checkpoint:", model_name)

AraBERT preprocessor initialized successfully.
Model checkpoint: aubmindlab/bert-base-arabertv02


In [ ]:
sample_tweet = train_df["tweet_text"].iloc[0]

processed_sample = arabert_prep.preprocess(
    sample_tweet
)

print("Original tweet:")
print(sample_tweet)

print("\nAfter AraBERT preprocessing:")
print(processed_sample)

Original tweet:
محمد صلاح عبر تويتر: دايمًا سعيد لما بشوف الراجل ده (حجازي) 😀💗. https://t.co/flXjQmLOOH

After AraBERT preprocessing:
محمد صلاح عبر تويتر : دايما سعيد لما بشوف الراجل ده ( حجازي ) . [رابط]


In [ ]:
train_df["processed_text"] = train_df[
    "tweet_text"
].apply(
    arabert_prep.preprocess
)

print("Training data preprocessing completed.")

Training data preprocessing completed.


In [ ]:
validation_df["processed_text"] = validation_df[
    "tweet_text"
].apply(
    arabert_prep.preprocess
)

print("Validation data preprocessing completed.")

Validation data preprocessing completed.


In [ ]:
test_df["processed_text"] = test_df[
    "tweet_text"
].apply(
    arabert_prep.preprocess
)

print("Test data preprocessing completed.")

Test data preprocessing completed.


In [ ]:
train_df[
    [
        "tweet_text",
        "processed_text",
        "Sentiment_label"
    ]
].head()

,tweet_text,processed_text,Sentiment_label
0,محمد صلاح عبر تويتر: دايمًا سعيد لما بشوف الرا...,محمد صلاح عبر تويتر : دايما سعيد لما بشوف الرا...,Positive
1,وزير الخارجية يلتقي نظيره الفرنسي علي هامش فعا...,وزير الخارجية يلتقي نظيره الفرنسي علي هامش فعا...,Neutral
2,عندما أنظر إلى حال هذه الأمه ، يخطر ببالي مجمو...,عندما أنظر إلى حال هذه الأمه ، يخطر ببالي مجمو...,Negative
3,تاريخ #كرة_القدم في المحك الآن #كأس_العالم دون...,تاريخ # كرة _ القدم في المحك الآن # كأس _ العا...,Negative
4,الصحف الإسبانية تقارن محمد صلاح بنجم برشلونةht...,الصحف الإسبانية تقارن محمد صلاح بنجم برشلونة [...,Neutral


In [ ]:
for i in range(3):

    print("=" * 80)

    print("Original:")
    print(train_df.loc[i, "tweet_text"])

    print("\nProcessed:")
    print(train_df.loc[i, "processed_text"])

    print("\nSentiment:")
    print(train_df.loc[i, "Sentiment_label"])

    print()

Original:
محمد صلاح عبر تويتر: دايمًا سعيد لما بشوف الراجل ده (حجازي) 😀💗. https://t.co/flXjQmLOOH

Processed:
محمد صلاح عبر تويتر : دايما سعيد لما بشوف الراجل ده ( حجازي ) . [رابط]

Sentiment:
Positive

Original:
وزير الخارجية يلتقي نظيره الفرنسي علي هامش فعاليات منتدي شباب العالم بشرم الشيخ https://t.co/dIDUI5y1JX

Processed:
وزير الخارجية يلتقي نظيره الفرنسي علي هامش فعاليات منتدي شباب العالم بشرم الشيخ [رابط]

Sentiment:
Neutral

Original:
عندما أنظر إلى حال هذه الأمه ، يخطر ببالي مجموعة من أوراق الشجر تعبث بها رياح الخريف فيما يسمى «الربيع العربي»Bashar

Processed:
عندما أنظر إلى حال هذه الأمه ، يخطر ببالي مجموعة من أوراق الشجر تعبث بها رياح الخريف فيما يسمى « الربيع العربي » Bashar

Sentiment:
Negative



In [ ]:
train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

validation_dataset = Dataset.from_pandas(
    validation_df,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df,
    preserve_index=False
)

In [ ]:
prepared_dataset = DatasetDict({
    "train": train_dataset,
    "validation": validation_dataset,
    "test": test_dataset
})

print(prepared_dataset)

DatasetDict({
    train: Dataset({
        features: ['tweet_text', 'Sentiment_label', 'processed_text'],
        num_rows: 14872
    })
    validation: Dataset({
        features: ['tweet_text', 'Sentiment_label', 'processed_text'],
        num_rows: 1859
    })
    test: Dataset({
        features: ['tweet_text', 'Sentiment_label', 'processed_text'],
        num_rows: 1859
    })
})


In [ ]:
print("Final columns:")
print(
    prepared_dataset["train"].column_names
)

print("\nExample prepared training sample:")
print(
    prepared_dataset["train"][0]
)

Final columns:
['tweet_text', 'Sentiment_label', 'processed_text']

Example prepared training sample:
{'tweet_text': 'محمد صلاح عبر تويتر: دايمًا سعيد لما بشوف الراجل ده (حجازي) 😀💗. https://t.co/flXjQmLOOH', 'Sentiment_label': 'Positive', 'processed_text': 'محمد صلاح عبر تويتر : دايما سعيد لما بشوف الراجل ده ( حجازي ) . [رابط]'}


In [ ]:
print("=" * 60)
print(
    "PART 1: SETUP + DATA PREPARATION + PREPROCESSING COMPLETED"
)
print("=" * 60)

print("\nDataset splits:")
print("Train:", len(prepared_dataset["train"]))
print(
    "Validation:",
    len(prepared_dataset["validation"])

)
print("Test:", len(prepared_dataset["test"]))

print("\nFinal columns:")
print(
    prepared_dataset["train"].column_names
)

print("\nSentiment classes:")
print(
    sorted(
        train_df["Sentiment_label"].unique()
    )
)

print("\nNumber of sentiment classes:")
print(
    train_df["Sentiment_label"].nunique()
)

print(
    "\nNote: The ArSAS dataset contains "
    "Positive, Negative, and Neutral only."
)

PART 1: SETUP + DATA PREPARATION + PREPROCESSING COMPLETED

Dataset splits:
Train: 14872
Validation: 1859
Test: 1859

Final columns:
['tweet_text', 'Sentiment_label', 'processed_text']

Sentiment classes:
['Negative', 'Neutral', 'Positive']

Number of sentiment classes:
3

Note: The ArSAS dataset contains Positive, Negative, and Neutral only.


Label Encoding, Tokenization & Model Setup

In [ ]:
#Label encoding

labels = sorted(train_df["Sentiment_label"].unique())
label2id = {
    label: i for i,label in enumerate(labels)
}
id2label = {v:k for k,v in label2id.items()}
train_df['label'] = train_df['Sentiment_label'].map(label2id)
validation_df['label'] = validation_df['Sentiment_label'].map(label2id)
test_df['label'] = test_df['Sentiment_label'].map(label2id)


In [ ]:
#rebuild dataset after updates
prepared_dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(validation_df, preserve_index=False),
    "test": Dataset.from_pandas(test_df, preserve_index=False),
})

In [ ]:
#Tokenization
from transformers import AutoTokenizer
model_ckpt = "aubmindlab/bert-base-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

In [ ]:
def tokenize(batch):
    temp = tokenizer(batch['processed_text'], padding='max_length', truncation=True, max_length=128)
    return temp
print(tokenize(prepared_dataset['train'][:2]))

{'input_ids': [[2, 582, 3173, 1115, 8454, 31, 53006, 181, 2140, 1439, 635, 509, 14544, 182, 7896, 14, 24914, 15, 20, 5, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 902, 1262, 7349, 6112, 2429, 485, 5994, 3689, 1167, 458, 3001, 619, 50898, 1158, 5, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [ ]:
#running tokenization on the full dataset
dataset_encoded = prepared_dataset.map(tokenize, batched=True)
print(dataset_encoded)
print(dataset_encoded['train'].column_names)

Map:   0%|          | 0/14872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1859 [00:00<?, ? examples/s]

Map:   0%|          | 0/1859 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tweet_text', 'Sentiment_label', 'processed_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 14872
    })
    validation: Dataset({
        features: ['tweet_text', 'Sentiment_label', 'processed_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1859
    })
    test: Dataset({
        features: ['tweet_text', 'Sentiment_label', 'processed_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1859
    })
})
['tweet_text', 'Sentiment_label', 'processed_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']


In [ ]:
#removing columns not needed by the model
dataset_encoded = dataset_encoded.remove_columns(["tweet_text", "Sentiment_label", "processed_text"])
print(dataset_encoded['train'].column_names)

['label', 'input_ids', 'token_type_ids', 'attention_mask']


In [ ]:
#change dataset format
dataset_encoded.set_format("torch")
print(type(dataset_encoded['train'][0]['input_ids']))

<class 'torch.Tensor'>


In [ ]:
#Model setup
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)

print(model.config.id2label)
print(model.config.num_labels)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{0: 'Negative', 1: 'Neutral', 2: 'Positive'}
3
